# Graph Neural Network for DARPA Anomaly Detection
Fixed version addressing gradient collapse and class imbalance.

In [8]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv
from sklearn.metrics import classification_report, f1_score, roc_auc_score

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 1. Data Loading and Preprocessing

In [9]:
# Load the dataset
df = pd.read_csv("darpa_ground_truth.csv")

# Clean up column names just in case
df.columns = df.columns.str.strip()

print("Dataset Shape:", df.shape)
# print("Class Distribution:\n", df['label'].value_counts())

Dataset Shape: (4554343, 1)


In [10]:
# Map unique IPs to node IDs
unique_ips = pd.concat([df['source_ip'], df['destination_ip']]).unique()
ip_to_idx = {ip: idx for idx, ip in enumerate(unique_ips)}
num_nodes = len(unique_ips)

# Build edge index tensor (shape: [2, num_edges])
# edge_src = df['source_ip'].map(ip_to_idx).values
# edge_dst = df['destination_ip'].map(ip_to_idx).values
edge_index = torch.tensor(np.array([edge_src, edge_dst]), dtype=torch.long)

# Target labels for edges
tensor_y = torch.tensor(df['label'].values, dtype=torch.float).unsqueeze(1)

# Feature engineering: Simple node degree features as initial node attributes
node_degrees = torch.zeros(num_nodes, 1)
for src in edge_src:
    node_degrees[src] += 1

# Project to a standard feature dimension (e.g., 16) to pass into models
node_features = node_degrees.repeat(1, 16) 

# Create PyG Data object
data_load = Data(x=node_features, edge_index=edge_index)
print(f"Graph constructed with {num_nodes} nodes and {data_load.num_edges} edges.")

KeyError: 'source_ip'

## 2. Data Splitting & Stratified/Balanced Masking

In [ ]:
# 70/15/15 Data Split
num_edges = data_load.num_edges
indices = np.arange(num_edges)
np.random.shuffle(indices)

train_end = int(0.7 * num_edges)
val_end = int(0.85 * num_edges)

train_idx_raw = indices[:train_end]
val_idx = indices[train_end:val_end]
test_idx = indices[val_end:]

# CRITICAL FIX: Implement a balanced training mask step
# We undersample the majority training class ONLY for gradient computation
train_labels_raw = tensor_y[train_idx_raw].flatten().numpy()
pos_train_idx = train_idx_raw[train_labels_raw == 1]
neg_train_idx = train_idx_raw[train_labels_raw == 0]

# Sample negative training edges to match positive training edges (1:1 ratio)
np.random.shuffle(neg_train_idx)
balanced_neg_train_idx = neg_train_idx[:len(pos_train_idx)]

# Recombine into final balanced training index pool
balanced_train_idx = np.concatenate([pos_train_idx, balanced_neg_train_idx])
np.random.shuffle(balanced_train_idx)

# Assign masks to PyG structure
train_mask = torch.zeros(num_edges, dtype=torch.bool)
train_mask[balanced_train_idx] = True

val_mask = torch.zeros(num_edges, dtype=torch.bool)
val_mask[val_idx] = True

test_mask = torch.zeros(num_edges, dtype=torch.bool)
test_mask[test_idx] = True

data_load.train_mask = train_mask
data_load.val_mask = val_mask
data_load.test_mask = test_mask

# Move fixed elements to execution device
data_load = data_load.to(device)
tensor_y = tensor_y.to(device)

print(f"Balanced Train Samples: {data_load.train_mask.sum().item()}")

NameError: name 'data_load' is not defined

## 3. GNN Model Architectures (Fixed Forward Passes)
*CRITICAL FIX: Removed `torch.sigmoid()` from the terminal step. Models now correctly output raw logits to accommodate stable `BCEWithLogitsLoss` updates.*

In [ ]:
class BaseModel(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(BaseModel, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        # Edge predictor layer: src features + dst features -> prediction logit
        self.edge_pred = nn.Linear(hidden_channels * 2, 1)
 
    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        
        # Gather source and destination node embeddings for all edges
        src, dst = edge_index[0], edge_index[1]
        edge_features = torch.cat([h[src], h[dst]], dim=1)
        
        # Return RAW LOGITS. Do NOT use torch.sigmoid() here.
        return self.edge_pred(edge_features)

class AdvancedModel(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(AdvancedModel, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.edge_pred = nn.Linear(hidden_channels * 2, 1)
 
    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        src, dst = edge_index[0], edge_index[1]
        edge_features = torch.cat([h[src], h[dst]], dim=1)
        return self.edge_pred(edge_features)

## 4. Custom Focal Loss Implementation

In [ ]:
class FocalLossWithLogits(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super(FocalLossWithLogits, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
 
    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss
        return focal_loss.mean()

## 5. Refactored Train & Evaluation Engine

In [ ]:
def train_and_evaluate(model, data, labels, epochs=30, lr=0.01):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = FocalLossWithLogits(alpha=0.6, gamma=2.0)
    
    print(f"\n--- Training {model.__class__.__name__} ---")
    
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        
        out = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], labels[data.train_mask])
        
        loss.backward()
        optimizer.step()
        
        if epoch % 5 == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                val_out = model(data.x, data.edge_index)
                val_preds = (val_out[data.val_mask] >= 0.0).float()
                val_f1 = f1_score(labels[data.val_mask].cpu(), val_preds.cpu(), average='macro', zero_division=0)
            print(f"Epoch {epoch:02d} | Train Loss: {loss.item():.4f} | Val Macro F1: {val_f1:.4f}")
 
    model.eval()
    with torch.no_grad():
        test_out = model(data.x, data.edge_index)
        test_logits = test_out[data.test_mask]
        
        test_preds = (test_logits >= 0.0).float().cpu().numpy()
        test_targets = labels[data.test_mask].cpu().numpy()
        test_probs = torch.sigmoid(test_logits).cpu().numpy()
        
        print(f"\n[Final Evaluation Report for {model.__class__.__name__}]")
        print(classification_report(test_targets, test_preds, target_names=['Non-Anomaly (0)', 'Anomaly (1)'], zero_division=0))
        
        try:
            auc = roc_auc_score(test_targets, test_probs)
            print(f"ROC-AUC Score: {auc:.4f}")
        except ValueError:
            print("ROC-AUC Score: Not computable")

## 6. Run Configurations

In [ ]:
base_gnn = BaseModel(in_channels=16, hidden_channels=32)
advanced_gnn = AdvancedModel(in_channels=16, hidden_channels=32)

train_and_evaluate(base_gnn, data_load, tensor_y, epochs=20, lr=0.01)
train_and_evaluate(advanced_gnn, data_load, tensor_y, epochs=20, lr=0.01)